# 310 — Classification data-prep & feature cache

Builds the classification-ready feature matrices **once** and caches them to disk so the
experiment notebooks (`320`, `330`) load instantly and never re-walk the ERSP tree.

**Source** — the same per-electrode ERSP `.npy` files the clustering pipeline uses:
`01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY/<pid>/LM/ERSP_matrix/<cond>/`.

**Three feature variants** (identical definitions to 02's feature sets):
| variant | what | dims |
|---|---|---|
| `rawds` | band-aware downsampled ERSP (15 bands × 30 time) | 450 |
| `hg` | high-gamma 70–150 Hz band-mean time series | 300 |
| `hg_ds` | the HG series downsampled to 30 time bins | 30 |

**What it caches**
- **Condition task** (gated): one sample per high-activity electrode × condition.
- **Parcellation task**: one sample per electrode = `audio ⊕ picture ⊕ reading`
  concatenated, for each variant × {Yeo-7, Yeo-17}. An electrode is kept iff it has all
  three conditions, is high-activity in ≥1 condition, and sits in a real Yeo network
  (medial-wall / white-matter / unknown contacts are dropped).

Only patients with **all three** conditions enter either task. Run this first, then 320 & 330.


In [1]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, str(Path('..').resolve()))
from functions import lf_classify as C

# Server UNC path first, local relative path as fallback (mirrors 02's notebooks).
INPUT_DIR = Path(r'\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY').resolve()

print('INPUT_DIR :', INPUT_DIR, '| exists:', INPUT_DIR.exists())
print('CACHE     :', C.DATASET_CACHE)
print('COORDS    :', C.COORDS_DIR, '| exists:', C.COORDS_DIR.exists())
print('variants  :', C.VARIANTS)


INPUT_DIR : \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY | exists: True
CACHE     : \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\03_FBM_Classifying\outputs\_dataset\classification
COORDS    : \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\250_recon\fsaverage\coords | exists: True
variants  : ('rawds', 'hg', 'hg_ds')


## 1 — Load the full ungated dataset
Walks every electrode × condition ERSP for patients that have all three conditions. The
high-activity flag is computed per sample (so both tasks derive from this one object).
The heavy walk is itself cached by `prepare_dataset`, so re-running is cheap.


In [2]:
df_meta, X_3d = C.prepare_full_dataset(INPUT_DIR)
print('samples:', len(df_meta), '| X_3d:', X_3d.shape)
print('patients:', sorted(df_meta.patient_id.unique()))
df_meta.head()


[lf_dataset] detected 27 patient folders under \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY
  loaded 9049 samples
  excluded 177 non-neural channels → 8872 samples
[lf_dataset] canonical dataset ready: 8872 samples · X_3d.shape=(8872, 129, 300)
  cached to \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\03_FBM_Classifying\outputs\_dataset\classification\_raw_ungated
[prepare_full_dataset] kept 24 patients with all 3 conditions; dropped 2: ['EL044', 'PAT_3301']
  8587 electrode x condition samples · X_3d=(8587, 129, 300)
samples: 8587 | X_3d: (8587, 129, 300)
patients: ['EL030', 'EL033', 'EL035', 'EL036', 'EL037', 'EL038', 'EL040', 'EL042', 'EL043', 'EL045', 'PAT_2868', 'PAT_3066', 'PAT_3390', 'PAT_3415', 'PAT_3455', 'PAT_3780', 'PAT_3965', 'PAT_3975', 'PAT_5515', 'PAT_5533', 'PAT_6619', 'PAT_6684', 'PAT_6704', 'PAT_6854']


,patient_id,condition,task,electrode,file_path,prop_above_pos,prop_below_neg,high_activity,sample_idx,contact_norm
0,EL030,audio,LM,A_L10,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,0.096951,0.015995,True,0,AL10
1,EL030,audio,LM,A_L11,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,0.117080,0.018734,True,1,AL11
2,EL030,audio,LM,A_L12,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,0.133127,0.030982,True,2,AL12
3,EL030,audio,LM,A_L13,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,0.163643,0.060853,True,3,AL13
4,EL030,audio,LM,A_L14,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,0.155504,0.064031,True,4,AL14


## 2 — Condition task — cache the 3 feature variants
Gated electrode × condition samples, labelled by condition.


In [3]:
for v in C.VARIANTS:
    X, y, groups, meta, cols = C.build_condition_arrays(df_meta, X_3d, v)
    d = C.save_arrays('condition', None, v, X, y, groups, meta, cols)
    print('  saved ->', d)


[build_feature_matrix] variant=rawds -> X.shape=(1944, 450)
[condition:rawds] X=(1944, 450)  classes={'reading': 749, 'audio': 607, 'picture': 588}  patients=24
  saved -> \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\03_FBM_Classifying\outputs\_dataset\classification\condition\rawds
[build_feature_matrix] variant=hg -> X.shape=(1944, 300)
[condition:hg] X=(1944, 300)  classes={'reading': 749, 'audio': 607, 'picture': 588}  patients=24
  saved -> \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\03_FBM_Classifying\outputs\_dataset\classification\condition\hg
[build_feature_matrix] variant=hg_ds -> X.shape=(1944, 30)
[condition:hg_ds] X=(1944, 30)  classes={'reading': 749, 'audio': 607, 'picture': 588}  patients=24
  saved -> \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\03_FBM_Classifying\outputs\_dataset\classification\condition\hg_ds


## 3 — Parcellation task — cache 3 variants × {Yeo-7, Yeo-17}
One sample per electrode; features = the three conditions concatenated in fixed order
`[audio, picture, reading]`. Label = the electrode's Yeo network.


In [4]:
for n_net in (7, 17):
    for v in C.VARIANTS:
        X, y, groups, meta, cols = C.build_parcellation_arrays(
            df_meta, X_3d, v, n_networks=n_net)
        d = C.save_arrays('parcellation', f'yeo{n_net}', v, X, y, groups, meta, cols)
        print('  saved ->', d)


[parcel:yeo7:rawds] X=(958, 1350)  classes=7  patients=24
  dropped: missing_cond=8 no_activity=1693 no_label=82 non_network=124
  class counts: {'7Networks_7': 357, '7Networks_5': 181, '7Networks_2': 145, '7Networks_6': 114, '7Networks_4': 69, '7Networks_3': 52, '7Networks_1': 40}
  saved -> \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\03_FBM_Classifying\outputs\_dataset\classification\parcellation\yeo7\rawds
[parcel:yeo7:hg] X=(958, 900)  classes=7  patients=24
  dropped: missing_cond=8 no_activity=1693 no_label=82 non_network=124
  class counts: {'7Networks_7': 357, '7Networks_5': 181, '7Networks_2': 145, '7Networks_6': 114, '7Networks_4': 69, '7Networks_3': 52, '7Networks_1': 40}
  saved -> \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\03_FBM_Classifying\outputs\_dataset\classification\parcellation\yeo7\hg
[parcel:yeo7:hg_ds] X=(958, 90)  classes=7  patients=24
  dropped: missing_cond=8 no_activity=1693 no_label=82 non_network=124
  cl

## 4 — Summary
The cache under `outputs/_dataset/classification/` now holds every (task × variant) matrix.
320 and 330 read these directly — no need to re-run 310 unless the upstream ERSPs or the
high-activity / Yeo filters change.


In [5]:
rows = []
for v in C.VARIANTS:
    X, y, g, m, c = C.load_arrays('condition', None, v)
    rows.append(('condition', '-', v, str(X.shape), len(set(y)), len(set(g))))
for n_net in (7, 17):
    for v in C.VARIANTS:
        X, y, g, m, c = C.load_arrays('parcellation', f'yeo{n_net}', v)
        rows.append(('parcellation', f'yeo{n_net}', v, str(X.shape), len(set(y)), len(set(g))))
pd.DataFrame(rows, columns=['task', 'target', 'variant', 'X.shape', 'n_classes', 'n_patients'])


,task,target,variant,X.shape,n_classes,n_patients
0,condition,-,rawds,"(1944, 450)",3,24
1,condition,-,hg,"(1944, 300)",3,24
2,condition,-,hg_ds,"(1944, 30)",3,24
3,parcellation,yeo7,rawds,"(958, 1350)",7,24
4,parcellation,yeo7,hg,"(958, 900)",7,24
5,parcellation,yeo7,hg_ds,"(958, 90)",7,24
6,parcellation,yeo17,rawds,"(958, 1350)",17,24
7,parcellation,yeo17,hg,"(958, 900)",17,24
8,parcellation,yeo17,hg_ds,"(958, 90)",17,24
